# Feature Analysis & Anti-Leakage Verification## Evaluation of Rolling Historical and Calibration FeaturesValidates that:1. `previous_accuracy` never includes the current question's answer2. `mastery_score` reflects only past evidence using Laplace smoothing3. Group-aware splitting (`student_group_split`) prevents cross-set student contamination

In [ ]:
import pandas as pd
import numpy as np
from training.common import student_group_split

df_ednet = pd.read_parquet("../datasets/processed/ednet_kt1/ednet_kt1_normalized.parquet")

# Inspect a single student interaction sequence
sample_student = df_ednet[df_ednet["student_id"] == "ednet_u_101409"].sort_values(by="timestamp").head(8)
print("Sequential feature progression for student ednet_u_101409:")
print(sample_student[["question_id", "attempt_number", "previous_attempts", "previous_accuracy", "recent_accuracy", "mastery_score", "answer_correct"]])

### Anti-Leakage Verification: Target Independence

In [ ]:
# Correlate target with historical features on first attempts
first_attempts = df_ednet[df_ednet["attempt_number"] == 1]
print("Correlation between answer_correct and previous_accuracy (first interaction of student):")
print(first_attempts[["answer_correct", "previous_accuracy"]].corr())

### Group-Aware Train / Val / Test Split Verification

In [ ]:
train_df, val_df, test_df = student_group_split(df_ednet, student_col="student_id", test_size=0.20, val_size=0.15)

train_students = set(train_df["student_id"])
val_students = set(val_df["student_id"])
test_students = set(test_df["student_id"])

print(f"Train Students: {len(train_students):,}")
print(f"Val Students:   {len(val_students):,}")
print(f"Test Students:  {len(test_students):,}")

# Overlap checks
overlap_tv = train_students.intersection(val_students)
overlap_tt = train_students.intersection(test_students)
overlap_vt = val_students.intersection(test_students)

print(f"Train/Val Overlap: {len(overlap_tv)} (MUST BE 0)")
print(f"Train/Test Overlap: {len(overlap_tt)} (MUST BE 0)")
print(f"Val/Test Overlap: {len(overlap_vt)} (MUST BE 0)")
assert len(overlap_tv) == 0 and len(overlap_tt) == 0 and len(overlap_vt) == 0, 'Data leakage detected!'
print('SUCCESS: Zero student overlap across splits!')